In [ ]:
# Add description here
#

In [ ]:
# Uncomment the next two lines to enable auto reloading for imported modules
# %load_ext autoreload
# %autoreload 2
# For more info, see:
# https://docs.ploomber.io/en/latest/user-guide/faq_index.html#auto-reloading-code-in-jupyter

In [ ]:
# If this task has dependencies, list them them here
# (e.g. upstream = ['some_task']), otherwise leave as None.
upstream = None

# This is a placeholder, leave it as None
product = None
taxonomy_source = None
count_threshold = None
sampling_table_source = None
library_table_source = None
rank = None

## Load dependencies

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tqdm
import taxoniq
from seaborn import objects as so
import numpy as np
sns.set_style('ticks')
plt.rcParams['svg.fonttype'] = 'none'
import re

## Load data 

In [ ]:
def load_library_taxonomy_results(file, path, library):
    """
    assigning labels from reading https://mmseqs.com/latest/userguide.pdf, taxonomy output and TSV
    """
    u = pd.read_csv(
        path + '/' + file, 
        sep='\t', 
        header=None, 
        index_col=None,
        names=['seq_id', 'taxid', 'level', 'species_name', 'fragments', 'frag_labeled', 'agreement', 'support']
    )
    u['library'] = library

    return u[['library', 'seq_id', 'taxid', 'level', 'species_name', 'fragments', 'support']]


This function solves some minor format issues with the library names.

In [ ]:
def fix_numeric_format_issues(x):
    m = re.search("PV(\d{1,6})(\w{1,10})?", x)
    m = list(m.groups())
    if m[1] is not None:
        return 'PV{:03d}{:s}'.format(int(m[0]), m[1])
    else:
        return 'PV{:03d}'.format(int(m[0]))
    
print(fix_numeric_format_issues("PV03bgi"))
print(fix_numeric_format_issues("PV00003bgi"))
print(fix_numeric_format_issues("PV00003"))
print(fix_numeric_format_issues("PV00003asdasda"))

## Load reference 

The `.input` file is key here.

In [ ]:
input_reference = pd.read_csv("input-data/.input", header=None, names=["file"])
input_reference['library'] = input_reference['file'].apply(lambda x: fix_numeric_format_issues(x))
input_reference

The following loop loads all the data.

In [ ]:
taxonomy_output = []
for i, row in tqdm.tqdm(list(input_reference.iterrows())):
    tmp = load_library_taxonomy_results(
        row.file, 
        "input-data/",
        row.library
    )
    taxonomy_output.append(tmp)
taxonomy_output = pd.concat(taxonomy_output)
taxonomy_output

## Removing unclassified reads

Plotting classified vs unclassified reads.

In [ ]:
taxonomy_output['is_unclassified'] = taxonomy_output['taxid'] == 0
sns.catplot(
    taxonomy_output.value_counts(subset=['is_unclassified']).reset_index(), 
    x='is_unclassified', y='count', height=3.0, aspect=1.0, kind='bar'
)

Removing not-classified reads.

In [ ]:
taxonomy_output = taxonomy_output.query('is_unclassified == False').copy()

In [ ]:
g = sns.catplot(
    data=taxonomy_output.value_counts(subset=['level']).reset_index(),
    x='level', y='count', kind='bar', height=3.5
)
g.set_xlabels("classification rank")
g.set_xticklabels(rotation=90)

## Cross-referencing with environmental / library data

Loading McLeish supplementary tables 1 and 2, and cross-referencing our data. 

In [ ]:
mc24_table1 = pd.read_csv(sampling_table_source, sep=';')
mc24_table2 = pd.read_csv(library_table_source, sep=';')
mc24_table2 = mc24_table2.dropna(subset=['Library_code'])
mc24_table2['Collection_code'] = mc24_table2['Collection_code'].apply(lambda x: x.split("_")[0])
sample_reference = pd.merge(mc24_table1, mc24_table2, on='Collection_code').groupby(['Site_code', 'Collection_code', 'Library_code', 'Location', 'Host_taxon', 'Habitat', 'No_extracts'], as_index=False)['Date'].apply(lambda x: len(list(x)))
sample_reference

In [ ]:
taxonomy_output = pd.merge(taxonomy_output, sample_reference, left_on='library', right_on='Library_code')

## Cross-referencing taxonomy

Loading taxonomy

In [ ]:
taxonomy_reference = pd.read_json(taxonomy_source)

Merge

In [ ]:
taxonomy_output = pd.merge(taxonomy_output, taxonomy_reference, on='taxid')

## Counting hits by library / species

In [ ]:
taxonomy_output_counts = taxonomy_output.value_counts(
    subset=[
        'library', 'taxid', 'level', 'species_name', 'phylum', 'class', 'order', 'family', 'genus', 'species', 'Site_code', 
        'Collection_code', 'Host_taxon', 'Habitat'
    ], dropna=False
).reset_index()
taxonomy_output_counts

In [ ]:
g = sns.catplot(
    data=taxonomy_output_counts.value_counts(subset=['level']).reset_index(),
    x='level', y='count', kind='bar', height=3.5
)
g.set_xticklabels(rotation=90)

## Removing counts below `count_threshold`

In [ ]:
g = sns.displot(taxonomy_output_counts, x='count', height=3, bins=20, log_scale=(True, False))
g.axes[0,0].set_yscale('log')
g.axes[0,0].axvline(count_threshold, color="black", alpha=0.5, linewidth=1.5, linestyle='--')
g.set_xlabels("Contigs per hit", )
g.savefig(product['histogram_contigs_per_hit'])

In [ ]:
taxonomy_output_counts = taxonomy_output_counts.query(f'count > {count_threshold}')

In [ ]:
g = sns.catplot(
    data=taxonomy_output_counts.value_counts(subset=['level']).reset_index(),
    x='level', y='count', kind='bar', height=3.5
)
g.set_xticklabels(rotation=90)

## Filter by taxonomic rank

In [ ]:
category_level = ["no rank", "superkingdom", "phylum", "class", "order", "family", "genus", "species"]

In [ ]:
taxonomy_output_counts['taxonomic_category'] = taxonomy_output_counts['level'].apply(lambda x: category_level.index(x))
taxonomy_output_counts

In [ ]:
taxonomy_output_counts = taxonomy_output_counts.query('taxonomic_category >= {0}'.format(category_level.index(rank)))

In [ ]:
g = sns.catplot(
    data=taxonomy_output_counts.value_counts(subset=['level']).reset_index(),
    x='level', y='count', kind='bar', height=3.5
)
g.set_xticklabels(rotation=90)

## Saving

In [ ]:
taxonomy_output_counts.to_json(product['count'], orient='records')